# Paper 2 — Blocking Verification + Leak-Filter Audit

**Runtime: CPU, seconds.**

Two post-hoc checks that share the same inputs:

1. **Blocking verification** — P1 corrected parser, P2 full sensitivity reporting, P3 pool audit against the powered events.
2. **Leak-filter audit** — every comparison recomputed filtered vs unfiltered, flagging any whose verdict changes.

**Prerequisite:** `repaired_ground_truth.json` must already exist on Drive — run `ground_truth_repair.py` once first. Both scripts carry the old and corrected parsers inline, so nothing else needs importing.

The scope rows of both are skipped unless `SCOPE_ANALYSIS_ENABLED = True`.


In [ ]:
# ---- Google Drive -----------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
# ---- scope_flag (inlined; no file to place) ---------------------------------
# The analysis scripts below contain `import scope_flag` and gate their retired
# local/global SCOPE sections on it. Rather than editing that logic -- which
# would risk changing behaviour -- this cell installs a real module object of
# that name into sys.modules, so the imports below resolve to it unchanged.
#
# SET THIS ONE TOGGLE:
SCOPE_ANALYSIS_ENABLED = False   # False = paper path (default). True = also
                                 # run the retired scope analysis.
#
# NOTE this governs scope ANALYSIS AND REPORTING only, never sample
# construction. The event sets were drawn stratified on the human
# local/global labels and no flag can change that. See SCOPE_ISOLATION.md §3.

import sys, types

_sf = types.ModuleType("scope_flag")
_sf.SCOPE_ANALYSIS_ENABLED = SCOPE_ANALYSIS_ENABLED
_sf.SCOPE_BANNER_SHOWN = False

def _scope_banner():
    if _sf.SCOPE_BANNER_SHOWN:
        return _sf.SCOPE_ANALYSIS_ENABLED
    _sf.SCOPE_BANNER_SHOWN = True
    if _sf.SCOPE_ANALYSIS_ENABLED:
        print("\n[scope_flag] SCOPE_ANALYSIS_ENABLED=True — exploratory scope "
              "sections WILL run.")
        print("[scope_flag] These are retired/exploratory and are not reported "
              "in the paper.")
    else:
        print("\n[scope_flag] SCOPE_ANALYSIS_ENABLED=False — scope sections "
              "skipped (paper path).")
        print("[scope_flag] Set SCOPE_ANALYSIS_ENABLED=True above to run them.")
    return _sf.SCOPE_ANALYSIS_ENABLED

def _skip_note(section_name):
    print(f"\n[scope_flag] SKIPPED (retired scope analysis): {section_name}")
    print("[scope_flag]   re-enable by setting SCOPE_ANALYSIS_ENABLED = True above")

_sf.scope_banner = _scope_banner
_sf.skip_note = _skip_note
sys.modules["scope_flag"] = _sf
print(f"scope_flag installed. SCOPE_ANALYSIS_ENABLED = {SCOPE_ANALYSIS_ENABLED}")

> **Run the two sections in order, top to bottom.** They are separate scripts that define some helpers under the same names (`load_gt`, `normalise`, the parsers), so section 2 redefines section 1's. That is harmless when run in order — section 1 has already finished by then — but if you re-run section 1's final cell *after* running section 2, it will pick up section 2's definitions. Re-run from the top if you need to redo section 1.


## 1. Blocking verification (P1 / P2 / P3)


In [ ]:
# =============================================================================
# BLOCKING VERIFICATION — corrected parser, full sensitivity, pool audit
#
# Closes the three blocking items before the ARR version can lean on the
# steering validity result:
#
#   P1  The p = 0.033 headline was computed with the OLD extract_answer, which
#       had three confirmed bugs. The fix changed 7-9% of rollout verdicts.
#       Recompute everything with the corrected parser. This cuts both ways.
#
#   P2  The significance rule was changed after seeing the result. Report all
#       three unparseable treatments explicitly rather than a single verdict,
#       and show the old rule's answer alongside the new one so the reader can
#       judge.
#
#   P3  The false-positive audit was run against the pools named
#       counterfactual_pilot_events.json / filler_control_pilot_events.json.
#       The POWERED steering run's events must be verified directly -- the
#       filenames are shared but that was assumed, not checked.
#
# Also recomputes the LEXICAL comparison with the fixed parser for parity, so
# "lexical fails (p=0.127)" is not left standing on old-parser data.
#
# COST: CPU only, seconds. All rollout text is saved.
# =============================================================================



from pathlib import Path
import json
import re

import numpy as np
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

import scope_flag   # SCOPE_ANALYSIS_ENABLED gate; see SCOPE_ISOLATION.md

In [ ]:
GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"

STEER_REV = EXPORT_DIR / "steering_revision_rollouts_powered.json"
STEER_FIL = EXPORT_DIR / "steering_filler_rollouts_powered.json"
LEX_REV = EXPORT_DIR / "counterfactual_rollout_results.json"
LEX_FIL = EXPORT_DIR / "filler_control_rollout_results.json"
EVENTS_REV = EXPORT_DIR / "counterfactual_pilot_events.json"

In [ ]:
EVENTS_FIL = EXPORT_DIR / "filler_control_pilot_events.json"
REPAIRED_GT = EXPORT_DIR / "repaired_ground_truth.json"
OUT = EXPORT_DIR / "blocking_verification.json"


# ------------------------------------------------------- parsers ------------
def extract_answer_old(text):
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed[-1].strip()
    m = re.search(r'(?:answer is|therefore)[:\s]*([0-9A-Za-z\-\.\/]+)',
                  text, re.IGNORECASE)
    return m.group(1).strip() if m else "UNKNOWN"


def boxed_spans(text):
    spans = []
    for m in re.finditer(r'\\boxed\s*\{', text or ""):
        start = m.end() - 1
        depth = 0
        for j in range(start, len(text)):
            if text[j] == '{':
                depth += 1
            elif text[j] == '}':
                depth -= 1
                if depth == 0:
                    spans.append(text[start + 1:j])
                    break
    return spans


STOPWORDS = {
    "the", "a", "an", "is", "approximately", "about", "roughly", "exactly",
    "correct", "incorrect", "therefore", "thus", "so", "then", "that", "this",
    "it", "equal", "equals", "our", "final", "answer", "value", "result",
    "less", "more", "different", "same", "indeed",
}


def normalise_answer(s):
    if s is None:
        return "UNKNOWN"
    s = s.strip()
    s = s.replace("\u0120", " ").replace("\u010a", " ")   # BPE artefacts
    s = re.sub(r'^\$+|\$+$', '', s)
    s = re.sub(r'\\(?:left|right|,|;|!|quad|qquad)', '', s)
    s = re.sub(r'\\text\s*\{([^}]*)\}', r'\1', s)
    s = s.replace(r'\dfrac', r'\frac').replace(r'\tfrac', r'\frac')
    s = re.sub(r'\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}', r'\1/\2', s)
    s = re.sub(r'\s+', '', s)
    s = s.rstrip('.,;:')
    s = re.sub(r'^\{|\}$', '', s)
    return s or "UNKNOWN"

In [ ]:
def extract_answer_new(text):
    boxed = boxed_spans(text)
    if boxed:
        return normalise_answer(boxed[-1])
    for pat in (r'(?:final\s+)?answer\s*(?:is|=|:)\s*(.{0,40})',
                r'(?:therefore|thus|hence)\s*,?\s*(.{0,40})'):
        for m in re.finditer(pat, text, re.IGNORECASE):
            for tok in re.findall(r'[-+]?\$?\\?[0-9A-Za-z\\{}\^_/\.,]+', m.group(1)):
                cand = normalise_answer(tok)
                if cand == "UNKNOWN" or cand.lower() in STOPWORDS:
                    continue
                if re.search(r'[0-9]|\\', cand):
                    return cand
    return "UNKNOWN"


# ------------------------------------------------------- scoring ------------
def load_gt(use_repaired):
    """Ground truth. If use_repaired, prefer the dataset-derived answers from
    the audit over the Stage-1 values, which were truncated for 71 problems."""
    gt = {}
    for p in (EVENTS_REV, EVENTS_FIL):
        if p.exists():
            for e in json.load(open(p)):
                gt[(e["model"], e["problem_id"])] = e["answer_gt"]
    n_repaired = 0
    if use_repaired and REPAIRED_GT.exists():
        rep = json.load(open(REPAIRED_GT))
        for k, v in rep.items():
            model, pid = k.split("|", 1)
            if (model, pid) in gt and v.get("true_answer_gt"):
                gt[(model, pid)] = v["true_answer_gt"]
                n_repaired += 1
    return gt, n_repaired


def necessity_scores(path, gt, parser, unknown_mode):
    """unknown_mode: 'as_incorrect' | 'dropped' | 'as_correct'."""
    out = []
    for r in json.load(open(path)):
        g = gt.get((r["model"], r["problem_id"]))
        texts = r.get("rollout_texts", [])
        if g is None or not texts:
            continue
        vals = []
        for t in texts:
            a = parser(t)
            unk = (a == "UNKNOWN")
            if unk and unknown_mode == "dropped":
                continue
            if unk:
                correct = (unknown_mode == "as_correct")
            else:
                correct = (normalise_answer(a) == normalise_answer(g))
            vals.append(correct)
        if vals:
            out.append({"model": r["model"], "problem_id": r["problem_id"],
                        "human_label": r.get("human_label"),
                        "necessity": 1.0 - sum(vals) / len(vals)})
    return out


def compare(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 3 or len(b) < 3:
        return None
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    rng = np.random.RandomState(0)
    d = [rng.choice(a, len(a), True).mean() - rng.choice(b, len(b), True).mean()
         for _ in range(5000)]
    return {"p": float(p), "auc": float(u / (len(a) * len(b))),
            "mean_diff": float(a.mean() - b.mean()),
            "mean_a": float(a.mean()), "mean_b": float(b.mean()),
            "ci95": [float(x) for x in np.percentile(d, [2.5, 97.5])],
            "n_a": len(a), "n_b": len(b)}


MODES = [("as_incorrect", "as INCORRECT (defensible)"),
         ("dropped", "DROPPED as missing (defensible)"),
         ("as_correct", "as CORRECT (stress test)")]
DEFENSIBLE = {"as_incorrect", "dropped"}


def run_validity(rev_path, fil_path, label, gt, parser):
    print(f"\n  {label}")
    print(f"    {'treatment':<34}{'rev':>7}{'fil':>8}{'diff':>9}{'p':>10}  95% CI")
    table = {}
    for mode, desc in MODES:
        rev = necessity_scores(rev_path, gt, parser, mode)
        fil = necessity_scores(fil_path, gt, parser, mode)
        r = compare([s["necessity"] for s in rev], [s["necessity"] for s in fil])
        if r is None:
            continue
        star = "*" if r["p"] < 0.05 else " "
        print(f"    {desc:<34}{r['mean_a']:>7.3f}{r['mean_b']:>8.3f}"
              f"{r['mean_diff']:>+9.3f}{r['p']:>10.4f}{star} "
              f"[{r['ci95'][0]:+.3f}, {r['ci95'][1]:+.3f}]")
        table[mode] = r
    n_def = sum(1 for m in DEFENSIBLE
                if m in table and table[m]["p"] < 0.05 and table[m]["mean_diff"] > 0)
    n_all = sum(1 for m, r in table.items() if r["p"] < 0.05 and r["mean_diff"] > 0)
    print(f"    -> significant & positive: {n_def}/2 defensible, {n_all}/3 overall")
    print(f"       original all-three rule would say: "
          f"{'ROBUST' if n_all == 3 else 'PARTIAL' if n_all else 'FRAGILE'}")
    print(f"       two-defensible rule says:          "
          f"{'ROBUST' if n_def == 2 else 'PARTIAL' if n_def == 1 else 'FRAGILE'}")
    table["_verdict_all_three"] = n_all
    table["_verdict_defensible"] = n_def
    return table

In [ ]:
def run_scope(rev_path, gt, parser, mode="as_incorrect"):
    rev = necessity_scores(rev_path, gt, parser, mode)
    loc = [s["necessity"] for s in rev if s["human_label"] == "local"]
    glo = [s["necessity"] for s in rev if s["human_label"] == "global"]
    if len(loc) < 3 or len(glo) < 3:
        return None
    auc = roc_auc_score([1] * len(glo) + [0] * len(loc), glo + loc)
    _, p = mannwhitneyu(glo, loc, alternative="two-sided")
    q1, q2 = auc / (2 - auc), 2 * auc ** 2 / (1 + auc)
    se = np.sqrt((auc * (1 - auc) + (len(glo) - 1) * (q1 - auc ** 2)
                  + (len(loc) - 1) * (q2 - auc ** 2)) / (len(glo) * len(loc)))
    return {"auc": float(auc), "p": float(p), "n_local": len(loc), "n_global": len(glo),
            "ci95": [float(max(0, auc - 1.96 * se)), float(min(1, auc + 1.96 * se))]}


def main():
    results = {}

    # ---------------- P3: pool audit against the POWERED events -------------
    print("\n" + "#" * 78)
    print("# P3 — FALSE-POSITIVE AUDIT AGAINST THE ACTUAL POWERED EVENT POOLS")
    print("#" * 78)
    rep = json.load(open(REPAIRED_GT)) if REPAIRED_GT.exists() else {}
    fps = {k for k, v in rep.items() if v.get("correct_old") and not v.get("correct_new")}
    print(f"\n  false positives identified in audit: {len(fps)}")

    pools = {
        "steering revision (POWERED, as run)": STEER_REV,
        "steering filler  (POWERED, as run)": STEER_FIL,
        "lexical revision (as run)": LEX_REV,
        "lexical filler   (as run)": LEX_FIL,
    }
    p3 = {}
    for name, path in pools.items():
        if not path.exists():
            print(f"  {name:<38} file missing")
            continue
        keys = {f"{r['model']}|{r['problem_id']}" for r in json.load(open(path))}
        hit = sorted(keys & fps)
        p3[name] = {"n_events": len(keys), "n_false_positive": len(hit), "which": hit}
        flag = "  <-- CONTAMINATED" if hit else ""
        print(f"  {name:<38} {len(keys):>4} events, "
              f"{len(hit)} false positives{flag}")
        for h in hit:
            print(f"      {h}")
    results["p3_pool_audit"] = p3
    print("\n  NOTE: this checks the events ACTUALLY PRESENT in each results file,")
    print("  not the pilot event-list files. That was the gap in the earlier check.")

    # ---------------- P1/P2: recompute with corrected parser ----------------
    print("\n" + "#" * 78)
    print("# P1/P2 — VALIDITY UNDER OLD vs CORRECTED PARSER, ALL TREATMENTS")
    print("#" * 78)

    gt_old, _ = load_gt(use_repaired=False)
    gt_new, n_rep = load_gt(use_repaired=True)
    print(f"\n  ground truth: {n_rep} entries replaced with dataset-derived answers")

    print("\n" + "=" * 78)
    print("STEERING")
    print("=" * 78)
    results["steering_old_parser"] = run_validity(
        STEER_REV, STEER_FIL, "OLD parser, OLD ground truth (as reported: p=0.033)",
        gt_old, extract_answer_old)
    results["steering_new_parser"] = run_validity(
        STEER_REV, STEER_FIL, "CORRECTED parser + repaired ground truth",
        gt_new, extract_answer_new)

    print("\n" + "=" * 78)
    print("LEXICAL (parity check -- the p=0.127 number is also on old-parser data)")
    print("=" * 78)
    results["lexical_old_parser"] = run_validity(
        LEX_REV, LEX_FIL, "OLD parser, OLD ground truth (as reported: p=0.127)",
        gt_old, extract_answer_old)
    results["lexical_new_parser"] = run_validity(
        LEX_REV, LEX_FIL, "CORRECTED parser + repaired ground truth",
        gt_new, extract_answer_new)

    # ---------------- scope test, both parsers ------------------------------
    # RETIRED / EXPLORATORY -- gated. P1/P2/P3 above are the kept verification
    # items and run unconditionally. See SCOPE_ISOLATION.md.
    if scope_flag.scope_banner():
        print("\n" + "#" * 78)
        print("# SCOPE TEST — necessity vs human local/global")
        print("#   [RETIRED EXPLORATORY ANALYSIS — not reported in the paper]")
        print("#" * 78)
        for tag, gt, parser in (("old parser", gt_old, extract_answer_old),
                                ("corrected", gt_new, extract_answer_new)):
            s = run_scope(STEER_REV, gt, parser)
            if s:
                print(f"  steering, {tag:<12} AUC {s['auc']:.3f}  "
                      f"CI [{s['ci95'][0]:.3f}, {s['ci95'][1]:.3f}]  p={s['p']:.4f}  "
                      f"(n {s['n_local']}L/{s['n_global']}G)")
                results[f"scope_{tag.replace(' ', '_')}"] = s
    else:
        scope_flag.skip_note("scope test — necessity vs human local/global, both parsers")

    # ---------------- read --------------------------------------------------
    print("\n" + "#" * 78)
    print("# READ")
    print("#" * 78)
    new = results.get("steering_new_parser", {})
    old = results.get("steering_old_parser", {})
    if new and old:
        n_def_new = new.get("_verdict_defensible", 0)
        p_new = new.get("as_incorrect", {}).get("p")
        p_old = old.get("as_incorrect", {}).get("p")
        print(f"  steering validity, headline treatment: "
              f"p {p_old:.4f} (old) -> {p_new:.4f} (corrected)")
        if n_def_new == 2:
            print("  HOLDS under the corrected parser in both defensible treatments.")
            print("  The 'steering passes validity, lexical fails' framing survives.")
        elif n_def_new == 1:
            print("  WEAKENED: significant in only one defensible treatment now.")
            print("  Report as provisional; do not lead the paper on it.")
        else:
            print("  DOES NOT SURVIVE the parser fix. The 'steering passes' claim must be")
            print("  withdrawn and the framing reverts toward 'both mechanisms marginal'.")
    print("\n  Report the full sensitivity tables above, both rule verdicts, and the")
    print("  old-vs-corrected comparison. Do not report a single verdict.")

    json.dump(results, open(OUT, "w"), indent=2, default=str)
    print(f"\nWrote {OUT}")


if __name__ == "__main__":
    main()

## 2. Leak-filtering audit


In [ ]:
# =============================================================================
# LEAK-FILTERING AUDIT — every comparison, filtered vs unfiltered
#
# WHY
# Leak-filtering has corrupted two analyses in this project, both found by
# accident rather than by checking:
#
#   1. Applying the lexical ban list to STEERING output destroyed 53% of the
#      run (556/482 rollouts, 48/39 whole events). Steering bans no tokens, so
#      a steered rollout containing "wait" has not leaked past anything.
#   2. The lexical validity comparison gave p = 0.127 filtered and p = 0.0036
#      unfiltered. The filter washed out a real effect and produced a wrong
#      headline that a whole reframing was built on.
#
# The filter is non-random: it selects for rollouts that happen not to contain
# common discourse words. Applying it to one arm of a comparison and not the
# other, or to a mechanism with no active ban, biases the result in an
# unknown direction.
#
# WHAT THIS DOES
# Recomputes every comparison the paper relies on, both filtered and
# unfiltered, and flags any whose VERDICT changes. Anything flagged needs the
# unfiltered number reported, or an explicit justification for filtering.
#
# COST: CPU only, seconds.
# =============================================================================



from pathlib import Path
import json
import re

import numpy as np
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.metrics import roc_auc_score

In [ ]:
import scope_flag   # SCOPE_ANALYSIS_ENABLED gate; see SCOPE_ISOLATION.md

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"

FILES = {
    "steering_rev": EXPORT_DIR / "steering_revision_rollouts_powered.json",
    "steering_fil": EXPORT_DIR / "steering_filler_rollouts_powered.json",
    "lexical_rev": EXPORT_DIR / "counterfactual_rollout_results.json",
    "lexical_fil": EXPORT_DIR / "filler_control_rollout_results.json",
}
EVENTS_REV = EXPORT_DIR / "counterfactual_pilot_events.json"
EVENTS_FIL = EXPORT_DIR / "filler_control_pilot_events.json"

In [ ]:
REPAIRED_GT = EXPORT_DIR / "repaired_ground_truth.json"
OUT = EXPORT_DIR / "leak_filter_audit.json"

# The ban list. For LEXICAL arms these were actively suppressed during
# generation, so a hit is a genuine leak. For STEERING arms nothing was banned,
# so a hit is ordinary output and filtering on it is not meaningful.
BANNED = [
    "wait", "actually", "let me reconsider", "that's wrong", "that is wrong",
    "i made an error", "let me try again", "this doesn't work",
    "this does not work", "i need to reconsider", "hmm", "that's not right",
    "that is not right", "let me rethink", "on second thought",
    "let me go back", "wait, let me", "actually, let me", "let me recheck",
    "let me re-check", "alternatively", "i made a mistake",
    "i think i made a mistake", "let me double check", "let me double-check",
    "hold on",
]


def is_leaked(t):
    tl = t.lower()
    return any(p in tl for p in BANNED)


# ---------------------------------------------------------------- parser ----
def boxed_spans(text):
    spans = []
    for m in re.finditer(r'\\boxed\s*\{', text or ""):
        i = m.end() - 1
        depth = 0
        for j in range(i, len(text)):
            if text[j] == '{':
                depth += 1
            elif text[j] == '}':
                depth -= 1
                if depth == 0:
                    spans.append(text[i + 1:j])
                    break
    return spans


STOPWORDS = {"the", "a", "an", "is", "approximately", "about", "roughly",
             "exactly", "correct", "incorrect", "therefore", "thus", "so",
             "then", "that", "this", "it", "equal", "equals", "our", "final",
             "answer", "value", "result", "less", "more", "different", "same",
             "indeed"}

In [ ]:
def normalise(s):
    if s is None:
        return "UNKNOWN"
    s = s.strip().replace("\u0120", " ").replace("\u010a", " ")
    s = re.sub(r'^\$+|\$+$', '', s)
    s = re.sub(r'\\(?:left|right|,|;|!|quad|qquad)', '', s)
    s = re.sub(r'\\text\s*\{([^}]*)\}', r'\1', s)
    s = s.replace(r'\dfrac', r'\frac').replace(r'\tfrac', r'\frac')
    s = re.sub(r'\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}', r'\1/\2', s)
    s = re.sub(r'\s+', '', s).rstrip('.,;:')
    return re.sub(r'^\{|\}$', '', s) or "UNKNOWN"


def extract(text):
    b = boxed_spans(text)
    if b:
        return normalise(b[-1])
    for pat in (r'(?:final\s+)?answer\s*(?:is|=|:)\s*(.{0,40})',
                r'(?:therefore|thus|hence)\s*,?\s*(.{0,40})'):
        for m in re.finditer(pat, text, re.IGNORECASE):
            for tok in re.findall(r'[-+]?\$?\\?[0-9A-Za-z\\{}\^_/\.,]+', m.group(1)):
                c = normalise(tok)
                if c != "UNKNOWN" and c.lower() not in STOPWORDS and re.search(r'[0-9]|\\', c):
                    return c
    return "UNKNOWN"


# ------------------------------------------------------------------ data ----
def load_gt():
    gt = {}
    for p in (EVENTS_REV, EVENTS_FIL):
        if p.exists():
            for e in json.load(open(p)):
                gt[(e["model"], e["problem_id"])] = e["answer_gt"]
    if REPAIRED_GT.exists():
        for k, v in json.load(open(REPAIRED_GT)).items():
            m, pid = k.split("|", 1)
            if (m, pid) in gt and v.get("true_answer_gt"):
                gt[(m, pid)] = v["true_answer_gt"]
    return gt


def scores(path, gt, filtered):
    """Necessity per event. filtered=True applies the leak filter."""
    out, n_drop_ro, n_drop_ev = [], 0, 0
    if not path.exists():
        return out, 0, 0
    for r in json.load(open(path)):
        g = gt.get((r["model"], r["problem_id"]))
        texts = r.get("rollout_texts", [])
        if g is None or not texts:
            continue
        keep = [t for t in texts if not (filtered and is_leaked(t))]
        n_drop_ro += len(texts) - len(keep)
        if not keep:
            n_drop_ev += 1
            continue
        vals = [extract(t) == normalise(g) for t in keep]
        out.append({"model": r["model"], "problem_id": r["problem_id"],
                    "human_label": r.get("human_label"),
                    "necessity": 1.0 - sum(vals) / len(vals)})
    return out, n_drop_ro, n_drop_ev


def mw(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 3 or len(b) < 3:
        return None
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    return {"p": float(p), "auc": float(u / (len(a) * len(b))),
            "mean_diff": float(a.mean() - b.mean()),
            "n_a": len(a), "n_b": len(b)}


def verdict(r):
    if r is None:
        return "n/a"
    return "SIG+" if (r["p"] < 0.05 and r["mean_diff"] > 0) else \
           "SIG-" if (r["p"] < 0.05) else "ns"

In [ ]:
def main():
    gt = load_gt()
    results = {}

    # ---------------- attrition ---------------------------------------------
    print("\n" + "#" * 84)
    print("# FILTER ATTRITION — how much does the filter remove from each arm?")
    print("#" * 84)
    print(f"\n  {'arm':<16}{'events':>8}{'rollouts dropped':>19}{'events lost':>14}  note")
    attrition = {}
    for name, path in FILES.items():
        unf, _, _ = scores(path, gt, filtered=False)
        fil, dro, dev = scores(path, gt, filtered=True)
        note = "no active ban -- filter meaningless" if name.startswith("steering") else ""
        print(f"  {name:<16}{len(unf):>8}{dro:>19}{dev:>14}  {note}")
        attrition[name] = {"n_unfiltered": len(unf), "rollouts_dropped": dro,
                           "events_lost": dev}
    results["attrition"] = attrition

    # ---------------- comparisons -------------------------------------------
    print("\n" + "#" * 84)
    print("# EVERY COMPARISON, FILTERED vs UNFILTERED")
    print("#" * 84)

    def validity(mech):
        rows = {}
        for filt in (False, True):
            rev, _, _ = scores(FILES[f"{mech}_rev"], gt, filt)
            fil, _, _ = scores(FILES[f"{mech}_fil"], gt, filt)
            rows["filtered" if filt else "unfiltered"] = mw(
                [s["necessity"] for s in rev], [s["necessity"] for s in fil])
        return rows

    def scope(mech):
        rows = {}
        for filt in (False, True):
            rev, _, _ = scores(FILES[f"{mech}_rev"], gt, filt)
            loc = [s["necessity"] for s in rev if s["human_label"] == "local"]
            glo = [s["necessity"] for s in rev if s["human_label"] == "global"]
            if len(loc) >= 3 and len(glo) >= 3:
                auc = roc_auc_score([1] * len(glo) + [0] * len(loc), glo + loc)
                _, p = mannwhitneyu(glo, loc, alternative="two-sided")
                rows["filtered" if filt else "unfiltered"] = {
                    "p": float(p), "auc": float(auc), "mean_diff": float(auc - 0.5),
                    "n_a": len(glo), "n_b": len(loc)}
            else:
                rows["filtered" if filt else "unfiltered"] = None
        return rows

    def mechanism():
        rows = {}
        for filt in (False, True):
            a, _, _ = scores(FILES["lexical_rev"], gt, filt)
            b, _, _ = scores(FILES["steering_rev"], gt, filt)
            ka = {(s["model"], s["problem_id"]): s["necessity"] for s in a}
            kb = {(s["model"], s["problem_id"]): s["necessity"] for s in b}
            sh = sorted(set(ka) & set(kb))
            if len(sh) >= 10:
                rho, p = spearmanr([ka[k] for k in sh], [kb[k] for k in sh])
                rows["filtered" if filt else "unfiltered"] = {
                    "p": float(p), "auc": float(rho), "mean_diff": float(rho),
                    "n_a": len(sh), "n_b": len(sh)}
            else:
                rows["filtered" if filt else "unfiltered"] = None
        return rows

    # The two scope rows are the RETIRED exploratory analysis and are included
    # only when SCOPE_ANALYSIS_ENABLED=1. The kept conclusions of this audit are
    # the validity rows and the mechanism-agreement row, which run
    # unconditionally. See SCOPE_ISOLATION.md.
    checks = [
        ("steering validity (rev vs fil)", validity("steering")),
        ("lexical validity (rev vs fil)", validity("lexical")),
    ]
    if scope_flag.scope_banner():
        checks += [
            ("steering scope AUC vs labels [RETIRED]", scope("steering")),
            ("lexical scope AUC vs labels [RETIRED]", scope("lexical")),
        ]
    else:
        scope_flag.skip_note("leak-filter audit rows: steering/lexical scope AUC vs labels")
    checks += [
        ("mechanism agreement (rho)", mechanism()),
    ]

    print(f"\n  {'comparison':<34}{'unfiltered':>26}{'filtered':>26}  changed?")
    flagged = []
    for name, rows in checks:
        u, f = rows.get("unfiltered"), rows.get("filtered")
        du, df = verdict(u), verdict(f)
        us = f"p={u['p']:.4f} d={u['mean_diff']:+.3f} n={u['n_a']}/{u['n_b']}" if u else "n/a"
        fs = f"p={f['p']:.4f} d={f['mean_diff']:+.3f} n={f['n_a']}/{f['n_b']}" if f else "n/a"
        changed = (du != df)
        if changed:
            flagged.append(name)
        print(f"  {name:<34}{us:>26}{fs:>26}  "
              f"{'YES  ' + du + ' -> ' + df if changed else 'no'}")
        results[name] = {"unfiltered": u, "filtered": f,
                         "verdict_unfiltered": du, "verdict_filtered": df,
                         "verdict_changed": changed}

    # ---------------- read ---------------------------------------------------
    print("\n" + "#" * 84)
    print("# READ")
    print("#" * 84)
    if flagged:
        print(f"\n  {len(flagged)} comparison(s) CHANGE VERDICT under filtering:")
        for n in flagged:
            print(f"    - {n}")
        print("\n  For each, report the UNFILTERED number. The filter is non-random: it")
        print("  selects for rollouts that happen not to contain common discourse words,")
        print("  so it biases in an unknown direction. Filtering is only defensible where")
        print("  there was an ACTIVE token ban and the aim is to drop rollouts that")
        print("  escaped it -- and even then, not when the comparison arm is unfiltered.")
    else:
        print("\n  No comparison changes verdict under filtering. The choice does not")
        print("  affect any reported conclusion.")
    print("\n  Regardless: steering arms must never be leak-filtered. Nothing was banned")
    print("  during steering generation, so there is no leak to filter -- applying the")
    print("  lexical ban list there previously destroyed 53% of the run.")

    json.dump(results, open(OUT, "w"), indent=2, default=str)
    print(f"\nWrote {OUT}")


if __name__ == "__main__":
    main()

## Optional — statistics regression fixture

Confirms the shared AUC / Mann-Whitney / Hanley-McNeil implementation reproduces its known-good values. Pure arithmetic, no data needed.


In [ ]:
# =============================================================================
# AUC / MANN-WHITNEY / CONFIDENCE-INTERVAL UTILITIES
#
# Formerly `scope_stats_utils.py`. Renamed and de-scoped during the scope
# isolation pass (see SCOPE_ISOLATION.md): the statistics in this file are
# generic two-sample rank comparisons with nothing scope-specific about them,
# but the old name and the old `local_scores` / `global_scores` argument names
# made them read as if they belonged to the scope analysis. They do not --
# they are used by the KEPT results (the matched-filler validity controls for
# both suppression mechanisms) as well as by the retired scope analysis.
#
# Shared so the statistical protocol is visibly ONE implementation rather than
# several copies that could silently drift apart.
#
# Public API: auc_and_mannwhitney, hanley_mcneil_ci, full_report, print_report.
# Arguments are `neg_scores` / `pos_scores` (the class scored 0 and the class
# scored 1). AUC is P(a random positive-class score exceeds a random
# negative-class score).
# =============================================================================

import numpy as np
from sklearn.metrics import roc_auc_score
from scipy.stats import mannwhitneyu


def auc_and_mannwhitney(neg_scores, pos_scores):
    """
    AUC = P(a random pos_scores value exceeds a random neg_scores value), plus
    the two-sided Mann-Whitney U p-value. Threshold-free and prevalence-robust.

    Returns a dict with auc, p_value, n_neg, n_pos, median_neg, median_pos --
    or None values if either class has fewer than 2 points (AUC is not
    reliably estimable below that).
    """
    n_neg, n_pos = len(neg_scores), len(pos_scores)
    if n_neg < 2 or n_pos < 2:
        return {"auc": None, "p_value": None, "n_neg": n_neg, "n_pos": n_pos,
                "median_neg": float(np.median(neg_scores)) if len(neg_scores) else None,
                "median_pos": float(np.median(pos_scores)) if len(pos_scores) else None}

    labels = [1] * n_pos + [0] * n_neg
    scores = list(pos_scores) + list(neg_scores)
    auc = roc_auc_score(labels, scores)
    _, p_value = mannwhitneyu(pos_scores, neg_scores, alternative="two-sided")

    return {
        "auc": float(auc), "p_value": float(p_value),
        "n_neg": n_neg, "n_pos": n_pos,
        "median_neg": float(np.median(neg_scores)),
        "median_pos": float(np.median(pos_scores)),
    }


def hanley_mcneil_ci(auc, n_pos, n_neg, confidence=0.95):
    """
    95% CI for an AUC via the Hanley & McNeil (1982) standard error formula.
    Used throughout this project instead of trusting an AUC point estimate on
    its own -- several results here looked promising as point estimates and
    turned out to have CIs comfortably spanning chance (0.5), which mattered
    for correct interpretation.

    Returns (lower, upper), clipped to [0, 1].
    """
    if auc is None or n_pos < 2 or n_neg < 2:
        return None, None
    q1 = auc / (2 - auc)
    q2 = (2 * auc ** 2) / (1 + auc)
    se = np.sqrt((auc * (1 - auc) + (n_pos - 1) * (q1 - auc ** 2)
                  + (n_neg - 1) * (q2 - auc ** 2)) / (n_pos * n_neg))
    z = 1.959964  # ~95% two-sided normal quantile
    lo = max(0.0, auc - z * se)
    hi = min(1.0, auc + z * se)
    return float(lo), float(hi)


def full_report(neg_scores, pos_scores, label=""):
    """Convenience wrapper: auc_and_mannwhitney + hanley_mcneil_ci, combined
    into one dict in the reporting format used across this project."""
    r = auc_and_mannwhitney(neg_scores, pos_scores)
    if r["auc"] is not None:
        lo, hi = hanley_mcneil_ci(r["auc"], r["n_pos"], r["n_neg"])
        r["ci_95"] = [lo, hi]
    else:
        r["ci_95"] = [None, None]
    r["label"] = label
    return r


def print_report(r, neg_name="neg", pos_name="pos"):
    print(f"\n{'='*60}\n{r.get('label','')}  "
          f"(n_{neg_name}={r['n_neg']}, n_{pos_name}={r['n_pos']})\n{'='*60}")
    if r["auc"] is None:
        print("  TOO FEW EVENTS IN ONE CLASS -- cannot compute AUC reliably.")
        return
    print(f"  median -- {neg_name}: {r['median_neg']:.3f}   "
          f"{pos_name}: {r['median_pos']:.3f}")
    print(f"  AUC: {r['auc']:.3f}   95% CI: [{r['ci_95'][0]:.3f}, {r['ci_95'][1]:.3f}]")
    print(f"  Mann-Whitney p-value: {r['p_value']:.4f}")


if __name__ == "__main__":
    # Numeric regression fixture. These two arrays are the real
    # steering-pilot necessity values from the retired scope analysis, kept
    # here ONLY because they are a known-good hand-checkable input/output pair
    # for this file's arithmetic. The assertion below is a regression check on
    # the AUC implementation, not a result the paper reports.
    fixture_neg = [0.0, 0.0, 0.25, 0.125, 0.0, 0.75, 1.0]
    fixture_pos = [0.75, 0.0, 0.0, 0.0, 0.25, 0.75, 0.0, 0.0]
    r = full_report(fixture_neg, fixture_pos, label="regression fixture")
    print_report(r)
    assert abs(r["auc"] - 0.411) < 0.01, "regression fixture AUC mismatch"
    lo, hi = r["ci_95"]
    assert abs(lo - 0.113) < 0.01 and abs(hi - 0.709) < 0.01, "CI regression mismatch"
    print("\nRegression fixture passed (AUC and CI both reproduce).")